# Tuned plain Logistic Regression
Owner: Anupama Singh

The plain Logistic Regression notebook uses the scikit-learn default `C = 1`. This notebook checks whether tuning `C` helps: same model (L2 penalty, lbfgs), same 706 students and folds, but `C` is chosen inside each training fold from a wider grid (0.001 to 100).

## 1. Setup and data
Same data as the paper and as the rest of the team: 706 labeled students with no missing values, the paper's 16 features, two targets. The folds are identical to Megha's Random Forest and XGBoost notebooks.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, mean_absolute_error
from sklearn.linear_model import LogisticRegression
# Data: local copy in data/ if present, otherwise download from the ECRI repository
LOCAL = Path("../data")
URL = "https://raw.githubusercontent.com/ecri-data/ECRI-Dataset/main/ECRI"
src = LOCAL if (LOCAL / "ECRIDataSet.csv").exists() else URL

# Outputs: the repo's results/ folder when run locally, ./results in Colab
RESULTS = Path("../results") if Path("../results").exists() else Path("results")
RESULTS.mkdir(exist_ok=True)

# The paper's 16 features (Table 3) and the two Woodcock targets
FEATURES = ["Gender", "Tx", "Age1b", "Tier2_N", "grp_rate", "rcmistot", "gnrl_fid",
            "TKPctCrct", "NWFcls", "NWFwrc", "ORFwc", "SAwrS", "SAsrS", "SAtoS",
            "RMwidRS", "RMwdaRS"]
TARGETS = {"Word Identification": "RMwidRS", "Word Attack": "RMwdaRS"}

df = pd.read_csv(f"{src}/ECRIDataSet.csv")
raw = pd.read_csv(f"{src}/ECRIRawData.csv", low_memory=False)

# Spring scores are used ONLY to compute the gain (regression target), never as features
spring = raw[raw["Time"] == 2.0].set_index("StuID")[list(TARGETS.values())].add_suffix("_spring")

# Labeled students with no missing feature values, as in the paper (706 students)
data = (df[df["RMwidRS_target"].notna()]
        .dropna(subset=FEATURES)
        .join(spring, on="StuID")
        .reset_index(drop=True))
for score in TARGETS.values():
    data[f"{score}_gain"] = data[f"{score}_spring"] - data[score]

X = data[FEATURES]
print(f"Students: {len(data)} | Schools: {data['SchlID'].nunique()} | Features: {len(FEATURES)}")

Students: 706 | Schools: 42 | Features: 16


## 2. The model
- Features are standardized inside each training fold.
- Same model as the plain notebook: `LogisticRegression` with the L2 penalty and `max_iter=2000`.
- `C` sets the penalty strength: a **smaller** `C` means a **stronger** penalty.
- `C` is **tuned inside each training fold** with an inner 4-fold grouped cross-validation (best ROC AUC) over 11 values from 0.001 to 100. The test fold is never used for this choice.

In [2]:
import warnings
# Newer scikit-learn versions (1.8+) announce renamed settings; the code below works on all versions
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", message="Inconsistent values", category=UserWarning)

MODEL_NAME = "Logistic Regression (tuned C)"
KIND = "classifier"
C_VALUES = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]
SETTINGS = {f"C={c}": float(c) for c in C_VALUES}


def build_model(name):
    return make_pipeline(StandardScaler(), LogisticRegression(C=SETTINGS[name], max_iter=2000))


def pick_setting(X_train, y_train, groups_train):
    # Inner 4-fold grouped CV on the training folds only (best ROC AUC); the test fold is never used
    scores = {name: cross_val_score(build_model(name), X_train, y_train, cv=GroupKFold(4),
                                    groups=groups_train, scoring="roc_auc").mean()
              for name in SETTINGS}
    return max(scores, key=scores.get)

## 3. 5-fold cross-validation
- **School split (main):** `GroupKFold(5)` on `SchlID`. Every school sits in exactly one test fold, so each student is scored by a model that never saw their school.
- **Student split:** `GroupKFold(5)` on `StuID`, so a school can appear in both training and test.

Each round trains on 4 folds (about 565 students) and predicts the 5th (about 141). The prediction for every student is saved so the evaluation notebook can compute all metrics and curves.

In [3]:
SPLITS = {"school": "SchlID", "student": "StuID"}


def cross_validate(score, split):
    """5-fold grouped cross-validation. Returns one row per student with the out-of-fold prediction."""
    groups = data[SPLITS[split]]
    y = data[f"{score}_target"] if KIND == "classifier" else data[f"{score}_gain"]
    rows = []
    for fold, (train_idx, test_idx) in enumerate(GroupKFold(n_splits=5).split(X, y, groups), start=1):
        setting = pick_setting(X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx])
        model = build_model(setting).fit(X.iloc[train_idx], y.iloc[train_idx])
        test = data.iloc[test_idx]
        prediction = model.predict_proba(test[FEATURES])[:, 1]    # probability of label 1
        pred_label = (prediction >= 0.5).astype(int)
        rows.append(pd.DataFrame({
            "split": split, "target": score, "model": MODEL_NAME, "fold": fold,
            "StuID": test["StuID"].values, "SchlID": test["SchlID"].values, "setting": setting,
            "true_label": test[f"{score}_target"].astype(int).values,
            "true_gain": test[f"{score}_gain"].values,
            "score": prediction,          # probability of label 1
            "pred_label": pred_label,
        }))
    return pd.concat(rows, ignore_index=True)


predictions = pd.concat([cross_validate(score, split)
                         for split in SPLITS for score in TARGETS.values()], ignore_index=True)
print("Predictions:", len(predictions), "rows (706 students x 2 targets x 2 splits)")

Predictions: 2824 rows (706 students x 2 targets x 2 splits)


## 4. Quick summary
Mean over the 5 folds (test folds only).

In [4]:
rows = []
for (split, score, fold), g in predictions.groupby(["split", "target", "fold"]):
    row = {"split": split, "target": score, "fold": fold,
           "ROC AUC": roc_auc_score(g["true_label"], g["score"]),
           "PR-AUC": average_precision_score(g["true_label"], g["score"]),
           "Accuracy": accuracy_score(g["true_label"], g["pred_label"])}
    rows.append(row)
fold_results = pd.DataFrame(rows)

summary = (fold_results.drop(columns="fold").groupby(["split", "target"])
           .agg(["mean", "std"]).round(3)
           .rename(index={v: k for k, v in TARGETS.items()}, level=1))
summary

ROC AUC        PR-AUC        Accuracy       
                               mean    std   mean    std     mean    std
split   target                                                          
school  Word Attack           0.615  0.039  0.694  0.079    0.613  0.064
        Word Identification   0.671  0.056  0.683  0.047    0.639  0.057
student Word Attack           0.632  0.032  0.707  0.042    0.622  0.045
        Word Identification   0.666  0.071  0.688  0.092    0.642  0.064

## 5. Setting chosen in each fold
Picked by the inner cross-validation inside each training fold.

In [5]:
(predictions.groupby(["split", "target", "fold"])["setting"].first()
 .unstack("fold").rename(index={v: k for k, v in TARGETS.items()}, level=1))

fold                             1      2     3      4      5
split   target                                               
school  Word Attack            C=3  C=100   C=3  C=100   C=30
        Word Identification  C=100   C=30   C=1  C=0.3    C=1
student Word Attack           C=10  C=0.1  C=10   C=30  C=0.3
        Word Identification  C=0.3  C=0.1  C=30  C=100    C=1

## 6. Tuned vs plain (C = 1)
Same students and folds. The plain results come from `anupama_logistic_regression_cv.ipynb`.

In [6]:
plain = pd.read_csv(RESULTS / "logistic_regression_cv_fold_results.csv")
metrics = ["Accuracy", "PR-AUC", "ROC AUC"]
compare = pd.concat({
    "plain (C=1)": plain.groupby(["split", "target"])[metrics].mean(),
    "tuned C": fold_results.groupby(["split", "target"])[metrics].mean(),
}, axis=1).round(3).rename(index={v: k for k, v in TARGETS.items()}, level=1)
compare

plain (C=1)                 tuned C               
                               Accuracy PR-AUC ROC AUC Accuracy PR-AUC ROC AUC
split   target                                                                
school  Word Attack               0.611  0.694   0.614    0.613  0.694   0.615
        Word Identification       0.640  0.684   0.671    0.639  0.683   0.671
student Word Attack               0.616  0.707   0.634    0.622  0.707   0.632
        Word Identification       0.637  0.688   0.667    0.642  0.688   0.666

**Result:** tuning `C` gives no real gain. Every difference is within about 0.01, far smaller than the fold-to-fold spread (about ±0.05), and the chosen `C` changes a lot from fold to fold. The plain model is not overfitting (train and test accuracy are close), so the default `C = 1` is kept.

## Save the results

In [7]:
predictions.to_csv(RESULTS / "logistic_regression_tuned_cv_predictions.csv", index=False)
fold_results.to_csv(RESULTS / "logistic_regression_tuned_cv_fold_results.csv", index=False)
print(f"Saved logistic_regression_tuned_cv_predictions.csv and logistic_regression_tuned_cv_fold_results.csv to {RESULTS}/")

Saved logistic_regression_tuned_cv_predictions.csv and logistic_regression_tuned_cv_fold_results.csv to ../results/
